<a href="https://colab.research.google.com/github/hail-members/llm-based-services/blob/main/Chapter3_llm_overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM 강의자료 실습 노트북

구성은 다음과 같습니다.

1. 데이터 수집: 웹 크롤링, API 호출, 공개 데이터 다운로드  
2. 텍스트 전처리: 정제, 파일 입출력, 형태소 분석, 불용어 제거, 표제어 추출  
3. Hugging Face 기반 분류 모델 실습: 토크나이저/모델 로드, Trainer로 간단한 파인튜닝

Colab 기준으로 작성했으며, 설치 셀은 처음 한 번만 실행하시면 됩니다.

## 0. 실행 전 준비

아래 셀들은 필요한 라이브러리를 설치합니다.  

In [6]:
# 기본 실습 라이브러리 설치
!pip -q -U install beautifulsoup4 requests pandas nltk transformers datasets accelerate evaluate scikit-learn

In [7]:
# KoNLPy 설치를 위한 Java 점검 및 설치
# Colab에서는 Java가 이미 들어 있는 경우가 많지만, 없으면 설치를 시도합니다.

import os
import shutil

if shutil.which("java") is None:
    !apt-get -qq update
    !apt-get -qq install -y openjdk-11-jdk-headless

# 환경에 따라 Java 경로가 다를 수 있으므로 둘 다 확인합니다.
if os.path.exists("/usr/lib/jvm/java-11-openjdk-amd64"):
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
elif os.path.exists("/usr/lib/jvm/java-17-openjdk-amd64"):
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

!pip -q install konlpy JPype1

In [8]:
# NLTK 자원 다운로드
# stopwords: 불용어 사전
# wordnet / omw-1.4: 표제어 추출(lemmatization)에 사용
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# 1. 데이터 수집

- BeautifulSoup + requests를 이용한 웹 크롤링
- requests를 이용한 API 호출
- pandas를 이용한 공개 데이터 다운로드


## 1-1. 웹 크롤링 예시

강의자료의 핵심은 다음 흐름입니다.

1. 웹 페이지 요청 보내기  
2. HTML 받기  
3. BeautifulSoup으로 파싱하기  
4. 원하는 태그만 골라서 텍스트 추출하기

In [14]:
import requests
from bs4 import BeautifulSoup

# 예제용으로 안정적인 테스트 페이지를 사용합니다.
url = "http://example.com"

# timeout을 주면 응답이 너무 늦을 때 무한 대기하는 일을 줄일 수 있습니다.
response = requests.get(url)

# 요청이 실패했으면 바로 예외를 발생시켜 문제를 빠르게 확인합니다.
response.raise_for_status()

# HTML 문서를 파싱합니다.
soup = BeautifulSoup(response.text, "html.parser")

# 강의자료와 동일하게 h1 태그를 추출합니다.
titles = soup.find_all("h1")

print("추출된 h1 태그 텍스트")
for idx, title in enumerate(titles, start=1):
    print(f"{idx}. {title.get_text(strip=True)}")

추출된 h1 태그 텍스트
1. Example Domain


## 1-2. API 이용 예시

실습에서는 먼저 **인증 없이도 바로 확인 가능한 공개 테스트 API**를 써보고,  
그 다음에 실제 서비스용 템플릿을 함께 적어 두겠습니다.

In [15]:
import requests

# 공개 테스트 API 예시
api_url = "https://jsonplaceholder.typicode.com/todos/1"
response = requests.get(api_url, timeout=10)

if response.status_code == 200:
    data = response.json()
    print("API 응답 예시:")
    print(data)
else:
    print(f"Error: {response.status_code}")

API 응답 예시:
{'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}


In [16]:
# 실제 서비스 API를 호출할 때는 보통 아래와 같은 형태를 사용합니다.
# 아직 실행하지 않도록 예시 템플릿으로만 남겨 둡니다.

# api_url = "https://api.example.com/data"
# headers = {
#     "Authorization": "Bearer YOUR_API_KEY"
# }
# response = requests.get(api_url, headers=headers, timeout=10)
#
# if response.status_code == 200:
#     data = response.json()
#     print(data)
# else:
#     print(f"Error: {response.status_code}")

## 1-3. 공개된 데이터 다운로드 예시

슬라이드에서는 `pd.read_csv()`로 온라인 CSV를 읽는 구조를 보여 주었습니다.  
실습에서는 공개 CSV 파일을 하나 읽어서 확인합니다.

In [17]:
import pandas as pd

# 강의자료의 data.csv placeholder를 실제 공개 CSV 주소로 바꾼 예시입니다.
data_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

df = pd.read_csv(data_url)

print("상위 5개 행")
print(df.head())

print("\n데이터 크기:", df.shape)
print("컬럼 목록:", df.columns.tolist())

상위 5개 행
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

데이터 크기: (150, 5)
컬럼 목록: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


# 2. 텍스트 전처리

강의자료에 나온 전처리 코드는 아이디어는 맞지만, 몇 군데는 그대로 실행되지 않습니다.

예를 들면 다음 문제가 있었습니다.

- `clean_text()` 함수의 들여쓰기가 깨져 있음  
- 정규표현식 문자열 앞에 raw string(`r""`)이 없어 가독성이 낮음  
- 파일 입출력 예시는 있지만, 실제로 읽을 파일이 준비되어 있지 않음  
- 한국어 토큰과 영어 stopwords 예제가 섞여 있어 초보자 기준으로 혼동될 수 있음

아래에서는 같은 내용을 **실행 가능한 형태**로 다시 정리했습니다.

## 2-1. 텍스트 클리닝 함수

강의자료의 핵심은 다음 세 단계입니다.

1. HTML 태그 제거  
2. 특수문자 제거  
3. 연속 공백 정리

In [18]:
import re

def clean_text(text: str) -> str:
    # 기본적인 텍스트 정제 함수
    # 1) HTML 태그 제거
    # 2) 한글/영문/숫자/공백만 남기기
    # 3) 연속된 공백을 하나로 줄이기
    text = re.sub(r"<.*?>", "", text)                   # HTML 태그 제거
    text = re.sub(r"[^가-힣a-zA-Z0-9 ]", "", text)      # 특수문자 제거
    text = re.sub(r"\s+", " ", text)                   # 다중 공백 제거
    return text.strip()

sample_text = "<p>안녕하세요!!   LLM 수업입니다 :)   2025년</p>"

print("원본 텍스트:")
print(sample_text)

print("\n정제 후 텍스트:")
print(clean_text(sample_text))

원본 텍스트:
<p>안녕하세요!!   LLM 수업입니다 :)   2025년</p>

정제 후 텍스트:
안녕하세요 LLM 수업입니다 2025년


## 2-2. 파일 입출력: 예제용 파일 만들기

슬라이드에는 `data.txt`를 바로 읽는 코드가 있었지만,  
실습을 위해서는 먼저 파일을 하나 만들어 두는 편이 자연스럽습니다.

In [19]:
sample_file_text = """
<html>
  <body>
    이 문서는   전처리 예제입니다!!!
    LLM과 NLP를 함께 다룹니다.
  </body>
</html>
"""

with open("data.txt", "w", encoding="utf-8") as file:
    file.write(sample_file_text)

print("data.txt 파일 생성 완료")

data.txt 파일 생성 완료


## 2-3. 파일 읽기

아래 셀은 슬라이드의 파일 읽기 코드를 거의 그대로 옮긴 버전입니다.

In [20]:
with open("data.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

print("읽어온 원시 텍스트:")
print(raw_text)

읽어온 원시 텍스트:

<html>
  <body>
    이 문서는   전처리 예제입니다!!!
    LLM과 NLP를 함께 다룹니다.
  </body>
</html>



## 2-4. 정제 후 파일로 다시 저장하기

이 셀은 슬라이드의 `cleaned_text = clean_text(raw_text)` 예제를 이어서 실행하는 부분입니다.

In [21]:
cleaned_text = clean_text(raw_text)

with open("cleaned_data.txt", "w", encoding="utf-8") as file:
    file.write(cleaned_text)

print("정제된 텍스트:")
print(cleaned_text)
print("\ncleaned_data.txt 파일 저장 완료")

정제된 텍스트:
이 문서는 전처리 예제입니다 LLM과 NLP를 함께 다룹니다

cleaned_data.txt 파일 저장 완료


## 2-5. 한국어 형태소 분석: KoNLPy의 Okt 사용

강의자료에서는 한국어를 띄어쓰기 기준으로만 나누기 어렵기 때문에  
형태소 단위 토큰화가 중요하다고 설명했습니다.

아래는 Okt 형태소 분석기를 이용한 가장 기본적인 예제입니다.

In [22]:
from konlpy.tag import Okt

okt = Okt()

korean_sentence = "자연어처리는 매우 흥미롭습니다"
tokens_ko = okt.morphs(korean_sentence)

print("원문:")
print(korean_sentence)

print("\n형태소 단위 토큰:")
print(tokens_ko)

원문:
자연어처리는 매우 흥미롭습니다

형태소 단위 토큰:
['자연어', '처리', '는', '매우', '흥미롭습니다']


## 2-6. 불용어 제거 예시

원래 슬라이드 코드는 영어 stopwords를 사용하는 예제였습니다.  
그래서 여기서는 영어 토큰 리스트를 따로 만들어서 설명합니다.

주의할 점은 다음과 같습니다.

- 한국어 형태소 결과에 영어 stopwords를 바로 적용하면 의미가 맞지 않습니다.
- 불용어 제거는 언어별로 사전이 다릅니다.

In [23]:
from nltk.corpus import stopwords

# 영어 예문 토큰
english_tokens = ["this", "is", "a", "simple", "example", "for", "llm", "practice"]

# 영어 불용어 사전 로드
english_stopwords = set(stopwords.words("english"))

# 불용어를 제외한 핵심 단어만 남깁니다.
filtered_tokens = [word for word in english_tokens if word.lower() not in english_stopwords]

# 다시 하나의 문자열로 합칠 수도 있습니다.
filtered_text = " ".join(filtered_tokens)

print("원본 토큰:", english_tokens)
print("불용어 제거 후:", filtered_tokens)
print("문장으로 합친 결과:", filtered_text)

원본 토큰: ['this', 'is', 'a', 'simple', 'example', 'for', 'llm', 'practice']
불용어 제거 후: ['simple', 'example', 'llm', 'practice']
문장으로 합친 결과: simple example llm practice


## 2-7. 정규화 및 표제어 추출

슬라이드에서는 `WordNetLemmatizer`를 이용한 표제어 추출 예제가 있었습니다.  
표제어 추출은 단어를 기본형에 가깝게 정리하는 과정입니다.

예를 들면 다음과 같은 기대를 할 수 있습니다.

- `cars` → `car`
- `studies` → `study` 또는 품사에 따라 다른 결과

품사 정보를 주지 않으면 기대와 다른 결과가 나올 수도 있으므로,  
아래에서는 기본형과 동사 기준 결과를 함께 비교합니다.

In [24]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

tokens_en = ["cars", "running", "mice", "studies"]

# 기본 설정(품사 정보 미지정)
normalized_default = [lemmatizer.lemmatize(word) for word in tokens_en]

# 동사로 간주했을 때
normalized_as_verbs = [lemmatizer.lemmatize(word, pos="v") for word in tokens_en]

print("원본 토큰:", tokens_en)
print("기본 lemmatize 결과:", normalized_default)
print("동사 기준 lemmatize 결과:", normalized_as_verbs)

원본 토큰: ['cars', 'running', 'mice', 'studies']
기본 lemmatize 결과: ['car', 'running', 'mouse', 'study']
동사 기준 lemmatize 결과: ['cars', 'run', 'mice', 'study']


## 2-8. 전처리 전체 흐름 한 번에 보기

강의자료의 여러 셀을 하나의 흐름으로 묶으면 아래처럼 이해할 수 있습니다.

1. 원문 준비  
2. 정제  
3. 토큰화  
4. 필요한 후처리

In [25]:
raw_document = """
<div>
자연어처리는   매우 재미있습니다!!!
NLP and LLM are powerful tools for text applications.
</div>
"""

cleaned_document = clean_text(raw_document)
ko_tokens = okt.morphs(cleaned_document)

print("원본 문서:")
print(raw_document)

print("\n정제된 문서:")
print(cleaned_document)

print("\n한국어 형태소 토큰:")
print(ko_tokens)

원본 문서:

<div>
자연어처리는   매우 재미있습니다!!!
NLP and LLM are powerful tools for text applications.
</div>


정제된 문서:
자연어처리는 매우 재미있습니다NLP and LLM are powerful tools for text applications

한국어 형태소 토큰:
['자연어', '처리', '는', '매우', '재미있습니다', 'NLP', 'and', 'LLM', 'are', 'powerful', 'tools', 'for', 'text', 'applications']


# 3. Hugging Face 기반 분류 모델 실습

강의자료의 파인튜닝 코드는 핵심 구조만 보여 주는 형태였습니다.

- `AutoTokenizer.from_pretrained(...)`
- `AutoModelForSequenceClassification.from_pretrained(...)`
- `TrainingArguments(...)`
- `Trainer(...)`
- `trainer.train()`

하지만 실제 실행을 위해서는 다음이 더 필요합니다.

- 데이터셋 로드
- 텍스트 토큰화
- `train_dataset`, `eval_dataset` 준비
- 평가 지표 정의

아래에서는 그 부분을 보완해서 **Colab에서 돌아가는 최소 실습 예제**로 구성합니다.

## 3-1. 라이브러리 불러오기

이 부분은 강의자료의 Hugging Face 예제를 확장한 것입니다.

In [26]:
import numpy as np
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

## 3-2. 예제 데이터셋 불러오기

영화 리뷰 감성 분류용 IMDB 데이터셋을 사용합니다.  
전체를 다 쓰면 시간이 길어질 수 있으므로, 실습에서는 일부만 사용합니다.

In [27]:
dataset = load_dataset("imdb")

# 실습 시간을 줄이기 위해 일부 샘플만 사용합니다.
small_train = dataset["train"].shuffle(seed=42).select(range(300))
small_test = dataset["test"].shuffle(seed=42).select(range(100))

print("train 샘플 수:", len(small_train))
print("test 샘플 수:", len(small_test))
print("\n예시 데이터:")
print(small_train[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

train 샘플 수: 300
test 샘플 수: 100

예시 데이터:
{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}


## 3-3. 토크나이저와 모델 불러오기

강의자료에는 `bert-base-uncased`가 예시로 들어 있었습니다.  
실습에서도 그대로 사용할 수 있지만, Colab에서 더 빠르게 돌리고 싶다면  
`distilbert-base-uncased`로 바꿔도 구조는 동일합니다.

In [28]:
# 강의자료와 동일한 계열의 예시
model_name = "bert-base-uncased"

# 실행 시간이 더 짧은 실습을 원하면 아래처럼 바꿔도 됩니다.
# model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,   # IMDB는 긍정/부정 2개 라벨
)

print("사용 모델:", model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


사용 모델: bert-base-uncased


## 3-4. 데이터 토큰화

모델은 문자열을 바로 받을 수 없기 때문에,  
토크나이저로 문자열을 `input_ids`, `attention_mask` 등의 형태로 바꿔야 합니다.

In [29]:
def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,   # 너무 긴 문장은 자릅니다.
        max_length=128,    # 실습 속도를 위해 길이를 제한합니다.
    )

tokenized_train = small_train.map(tokenize_batch, batched=True)
tokenized_test = small_test.map(tokenize_batch, batched=True)

# Trainer가 기대하는 형식으로 컬럼 이름을 맞춥니다.
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# 원본 텍스트 컬럼은 학습에는 필요 없으므로 제거합니다.
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

# PyTorch 텐서 형식으로 바꿉니다.
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train[0].keys())

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


## 3-5. 평가 지표 정의

여기서는 가장 단순하게 accuracy만 계산합니다.

In [30]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

## 3-6. TrainingArguments 설정

강의자료의 `TrainingArguments(output_dir="./results")`는 최소 골격이었고,  
실제로는 배치 크기, epoch 수, 로그 출력 방식 등을 함께 정하는 경우가 많습니다.

최근 Transformers 문서 기준으로는 `eval_strategy`를 사용하는 예제가 많습니다.

In [31]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,                # 실습이므로 1 epoch만 사용
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",             # epoch이 끝날 때마다 평가
    save_strategy="no",                # 실습이므로 체크포인트 저장 생략
    logging_steps=20,
    report_to="none",                  # wandb 등 외부 로깅 비활성화
)

## 3-7. Trainer 생성

슬라이드의 `Trainer(...)` 예제에서 비어 있던 `train_dataset`, `eval_dataset` 자리에  
실제로 준비한 데이터셋을 넣습니다.

최근 버전에서는 `tokenizer=` 대신 `processing_class=`를 사용하는 쪽이 권장됩니다.

In [32]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer 준비 완료")

Trainer 준비 완료


## 3-8. 학습 실행

실제로 학습을 돌리는 셀입니다.  
GPU 런타임이면 더 빠르고, CPU 런타임이면 시간이 조금 더 걸릴 수 있습니다.

In [33]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.717731,0.721489,0.470000


TrainOutput(global_step=38, training_loss=0.7111785286351254, metrics={'train_runtime': 434.3881, 'train_samples_per_second': 0.691, 'train_steps_per_second': 0.087, 'total_flos': 19733329152000.0, 'train_loss': 0.7111785286351254, 'epoch': 1.0})

## 3-9. 평가 실행

학습이 끝난 뒤 테스트 셋 일부에 대해 성능을 확인합니다.

In [34]:
eval_result = trainer.evaluate()
print(eval_result)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.721488893032074, 'eval_accuracy': 0.47, 'eval_runtime': 37.5595, 'eval_samples_per_second': 2.662, 'eval_steps_per_second': 0.346, 'epoch': 1.0}


## 3-10. 간단한 추론 해보기

학습한 모델에 직접 문장을 넣어서 예측 결과를 확인합니다.

In [35]:
sample_texts = [
    "I loved this movie. The story was engaging and emotional.",
    "This was boring and a waste of time.",
]

inputs = tokenizer(
    sample_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=-1).cpu().tolist()

label_map = {0: "negative", 1: "positive"}

for text, pred in zip(sample_texts, predictions):
    print(f"문장: {text}")
    print(f"예측 라벨: {label_map[pred]}")
    print("-" * 60)

문장: I loved this movie. The story was engaging and emotional.
예측 라벨: positive
------------------------------------------------------------
문장: This was boring and a waste of time.
예측 라벨: negative
------------------------------------------------------------


# 4. 정리

실습을 더 확장하려면 다음 순서로 이어 가시면 됩니다.

1. 내가 가진 도메인 문서로 전처리 파이프라인 바꾸기  
2. 공개 API 대신 실제 서비스 API 연결하기  
3. IMDB 대신 내 분류 데이터셋으로 바꾸기  
4. 분류가 아니라 생성형 태스크에 맞는 모델로 교체하기